# Task 1 — Grounded-SAM GPU reference

Run each cell in order with a Kaggle T4/P100 GPU and Internet enabled. This notebook uses the uploaded dataset and uploaded code datasets; it does not clone Git. Cell 5 is a mandatory smoke test; inspect its overlays before running Cell 6.

In [ ]:
DATASET_SOURCE = "ATTACHED"         # use the uploaded dataset ZIP/folder
GOOGLE_DRIVE_ID = "1a58rACcCFnaj0ZiDoNyGbDWXunRyWziv"
ATTACHED_DATASET_PATH = "/kaggle/input/datasets/jareen777/egg-dataset"  # extracted folder or uploaded .zip path

CODE_SOURCE = "ATTACHED"            # uploaded code dataset; no Git clone
ATTACHED_CODE_DIR = "/kaggle/input/egg-code"

CKPT = "/kaggle/working/checkpoints/sam_vit_h.pth"
OUT = "/kaggle/working/task1_gsam"

In [ ]:
%pip install -q opencv-python-headless tqdm pyyaml transformers accelerate
%pip install -q git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os, glob, shutil, zipfile, subprocess, requests, torch

assert torch.cuda.is_available(), "GPU is OFF. Turn on Accelerator -> GPU T4/P100 and restart."
print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT):
    subprocess.run(["wget", "-q", "-O", CKPT,
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"], check=True)
print("checkpoint MB:", round(os.path.getsize(CKPT) / 1e6))

DATA = "/kaggle/working/data"
if DATASET_SOURCE == "DRIVE":
    os.makedirs(DATA, exist_ok=True)
    z = "/kaggle/working/dataset.zip"
    if not os.path.exists(z):
        url = f"https://drive.usercontent.google.com/download?id={GOOGLE_DRIVE_ID}&export=download&confirm=t"
        r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
        with open(z, "wb") as f:
            for ch in r.iter_content(1 << 20):
                f.write(ch)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA)
else:
    attached = ATTACHED_DATASET_PATH
    if os.path.isdir(attached):
        zips = sorted(glob.glob(attached + "/**/*.zip", recursive=True))
        if zips:
            attached_zip = zips[0]
        else:
            attached_zip = None
    else:
        attached_zip = attached if attached.lower().endswith(".zip") else None
    if attached_zip:
        os.makedirs(DATA, exist_ok=True)
        with zipfile.ZipFile(attached_zip) as zf:
            zf.extractall(DATA)
    else:
        DATA = attached

imgs = sorted(p for p in glob.glob(DATA + "/**/*", recursive=True)
              if p.lower().endswith((".jpg", ".jpeg", ".png")))
print(DATA, "->", len(imgs), "images")
assert len(imgs) == 110, f"expected 110 images, found {len(imgs)}"

In [ ]:
import yaml

if CODE_SOURCE != "ATTACHED":
    raise ValueError("Set CODE_SOURCE to ATTACHED; this notebook does not clone Git.")
else:
    assert os.path.isdir(ATTACHED_CODE_DIR), f"uploaded code dataset not found: {ATTACHED_CODE_DIR}"
    hits = glob.glob(ATTACHED_CODE_DIR + "/**/scripts/run_task1.py", recursive=True)
    assert hits, f"scripts/run_task1.py not found under {ATTACHED_CODE_DIR}"
    src_root = os.path.dirname(os.path.dirname(hits[0]))
    REPO = "/kaggle/working/repo"
    shutil.copytree(src_root, REPO, dirs_exist_ok=True)
os.chdir(REPO)
for needed in ["scripts/run_task1.py", "src/grounded_sam.py", "configs/task1_gsam.yaml"]:
    assert os.path.exists(needed), f"missing {needed}"
os.makedirs("src", exist_ok=True)
if not os.path.exists("src/__init__.py"):
    open("src/__init__.py", "w").close()
cfg = yaml.safe_load(open("configs/task1_gsam.yaml"))
cfg["sam_checkpoint"] = CKPT
cfg["device"] = "cuda"
yaml.safe_dump(cfg, open("configs/task1_gsam_kaggle.yaml", "w"), sort_keys=False)
print(open("configs/task1_gsam_kaggle.yaml").read())
print(subprocess.run(["python", "scripts/run_task1.py", "--help"],
                     capture_output=True, text=True).stdout)

In [ ]:
import time
os.makedirs("/kaggle/working/smoke_in", exist_ok=True)
for p in imgs[:5]:
    shutil.copy(p, "/kaggle/working/smoke_in")

t0 = time.time()
!python scripts/run_task1.py --config configs/task1_gsam_kaggle.yaml --input /kaggle/working/smoke_in --output /kaggle/working/smoke_out --device cuda
print(f"smoke test: {time.time()-t0:.1f}s for 5 images")
!find /kaggle/working/smoke_out -maxdepth 2 | head -40

import matplotlib.pyplot as plt, cv2
ov = sorted(glob.glob("/kaggle/working/smoke_out/**/overlays/*.png", recursive=True))[:5]
fig, ax = plt.subplots(1, max(len(ov), 1), figsize=(4 * max(len(ov), 1), 4))
ax = ax if hasattr(ax, "__len__") else [ax]
for a, p in zip(ax, ov):
    a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.set_title(os.path.basename(p)[:18]); a.axis("off")
plt.show()

print("STOP: inspect that only eggs are highlighted before running Cell 6.")

In [ ]:
t0 = time.time()
!python scripts/run_task1.py --config configs/task1_gsam_kaggle.yaml --input $DATA --output $OUT --device cuda
print(f"full run: {time.time()-t0:.1f}s")

shutil.make_archive("/kaggle/working/task1_gsam_outputs", "zip", OUT)
print("zip MB:", round(os.path.getsize("/kaggle/working/task1_gsam_outputs.zip") / 1e6, 1))
!ls $OUT